[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mlnjsh/rl-basics/blob/main/Section_4_on_policy_constant_alpha_mc.ipynb)

<div style="text-align:center">
    <h1>
        On-policy Monte Carlo Control
    </h1>
</div>
<br>

<div style="text-align:center">
    <p>
        In this notebook we are going to implement one of the two major strategies that exist when learning a policy by interacting with the environment, called on-policy learning. The agent will perform the task from start to finish and based on the sample experience generated, update its estimates of the q-values of each state-action pair Q(s,a).
    </p>
</div>


<br>

In [ ]:
# === Setup: load the shared course modules =====================================
# The long environment-definition cell has been moved into two files that live
# next to this notebook: `envs.py` (the Maze environment) and `utils.py` (all the
# plotting / evaluation helpers). That keeps every lesson focused on the algorithm.
import sys, os
if 'google.colab' in sys.modules and not os.path.exists('envs.py'):
    # On Google Colab there are no local files, so install the pinned gym version
    # and download the two helper modules from the course repository.
    !pip install -qq gym==0.23.0 pygame
    !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/envs.py
    !wget -q https://raw.githubusercontent.com/mlnjsh/rl-basics/main/utils.py

from envs import Maze              # The 5x5 grid-world environment.
from utils import test_agent, plot_policy, plot_action_values


## Import the necessary software libraries:

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Initialize the environment

In [ ]:
env = Maze()

In [ ]:
frame = env.render(mode='rgb_array')
plt.axis('off')
plt.imshow(frame)

In [ ]:
print(f"Observation space shape: {env.observation_space.nvec}")
print(f"Number of actions: {env.action_space.n}")

## Define value table $Q(s, a)$

#### Create the $Q(s, a)$ table

In [ ]:
action_values = np.zeros(shape=(5, 5, 4))

#### Plot Q(s, a)

In [ ]:
plot_action_values(action_values)

## Define the policy $\pi(s)$

#### Create the policy $\pi(s)$

In [ ]:
def policy(state, epsilon=0.):
    if np.random.random() < epsilon:
        return np.random.randint(4)
    else:
        av = action_values[state]
        return np.random.choice(np.flatnonzero(av == av.max()))

#### Test the policy with state (0, 0)

In [ ]:
action = policy((0,0))
print(f"Action taken in state (0,0): {action}")

#### Plot the policy

In [ ]:
plot_policy(action_values, frame)

## Implement the algorithm

**Your task: build *constant-alpha* Monte Carlo control.**

### The idea, in plain language

This is on-policy Monte Carlo control again -- the agent plays full episodes with an
**epsilon-greedy** policy and learns from the **real returns** it observed. The one change
is **how** each `Q(s, a)` is updated.

The previous method set `Q(s, a)` to the *average of every return* ever seen. That needs
you to store all past returns and adapts slowly. **Constant-alpha MC** instead nudges the
estimate a **small fixed step** toward each new return:

```
Q(s, a)  <-  Q(s, a)  +  alpha * ( G - Q(s, a) )
```

- `G - Q(s, a)` is the **error** (how wrong the current estimate was), and
- **alpha** is the **learning rate / step size** (bigger = faster but noisier learning).

This is an exponentially weighted moving average: recent episodes count more, and you
never have to remember the whole history.

### Outline (fill in the TODOs below)

For each episode:

1. **Play a full episode** with the epsilon-greedy policy, saving every
   `(state, action, reward)`.
2. **Walk backwards**, computing the return `G = reward_t + gamma * G`.
3. **Update** each visited `Q(s, a)` with the constant-alpha rule above.

The only line that differs from the plain-average notebook is the update step -- see if you
can write it yourself first.

## Show results

#### Show resulting value table $Q(s, a)$

In [ ]:
plot_action_values(action_values)

#### Show resulting policy $\pi(\cdot|s)$

In [ ]:
plot_policy(action_values, frame)

#### Test the resulting agent

In [ ]:
test_agent(env, policy)

## Resources

[[1] Reinforcement Learning: An Introduction. Ch. 4: Dynamic Programming](https://web.stanford.edu/class/psych209/Readings/SuttonBartoIPRLBook2ndEd.pdf)